# Week 6 — Business Impact Analysis
## Supply Chain Intelligence Project

**Input:** All processed CSVs from Weeks 1–5  
**Output:** `business_impact_report.csv`, `kpi_summary.csv`, charts

### Objectives
1. Consolidate all findings into business KPIs
2. Quantify financial impact of late deliveries
3. Quantify supplier risk in dollar terms
4. ROI calculation — what does fixing the problem save?
5. Executive summary table
6. Build final business impact charts
7. Save everything for the Streamlit app & PDF report

---
## 1. Imports & Load All Data

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# ── Load all processed data ────────────────────────────────────
df          = pd.read_csv('../data/processed/cleaned_supply_chain.csv',
                          parse_dates=['Order Date', 'Ship Date'])
supplier    = pd.read_csv('../data/processed/supplier_risk_scores.csv')
tier_summary= pd.read_csv('../data/processed/supplier_tier_summary.csv')
forecast_6m = pd.read_csv('../data/processed/forecast_6month_summary.csv')
impact      = pd.read_csv('../data/processed/business_impact_summary.csv')

print('All data loaded ✓')
print(f'Main dataset  : {df.shape}')
print(f'Suppliers     : {len(supplier)} regions')
print(f'Forecast      : {len(forecast_6m)} months ahead')

All data loaded ✓
Main dataset  : (180519, 49)
Suppliers     : 23 regions
Forecast      : 3 months ahead


---
## 2. Master KPI Summary

In [2]:
# ── Compute all master KPIs ────────────────────────────────────
total_revenue       = df['Revenue'].sum()
total_orders        = len(df)
total_late          = df['Is_Late'].sum()
total_cancelled     = df['Is_Cancelled'].sum()
late_rate           = df['Is_Late'].mean()
cancel_rate         = df['Is_Cancelled'].mean()
revenue_at_risk     = df['Revenue_At_Risk'].sum()
avg_order_value     = df['Revenue'].mean()
avg_delay           = df['Delivery_Delay_Days'].mean()
avg_profit          = df['Order Profit Per Order'].mean()
total_profit        = df['Order Profit Per Order'].sum()
profit_margin       = total_profit / total_revenue * 100
date_range_days     = (df['Order Date'].max() - df['Order Date'].min()).days
daily_revenue       = total_revenue / date_range_days
high_risk_regions   = len(supplier[supplier['Risk_Tier'] == 'High Risk'])
total_regions       = len(supplier)
forecast_growth     = (forecast_6m['Forecast_Orders'].mean() - 
                       df.groupby(df['Order Date'].dt.to_period('M')).size().mean()) / \
                       df.groupby(df['Order Date'].dt.to_period('M')).size().mean() * 100

print('=' * 60)
print('  MASTER KPI SUMMARY — SUPPLY CHAIN INTELLIGENCE')
print('=' * 60)
print(f'  REVENUE')
print(f'  Total Revenue          : ${total_revenue:>12,.0f}')
print(f'  Total Profit           : ${total_profit:>12,.0f}')
print(f'  Profit Margin          : {profit_margin:>11.1f}%')
print(f'  Avg Order Value        : ${avg_order_value:>12,.2f}')
print(f'  Daily Revenue (avg)    : ${daily_revenue:>12,.0f}')
print()
print(f'  DELIVERY PERFORMANCE')
print(f'  Total Orders           : {total_orders:>12,}')
print(f'  Late Orders            : {total_late:>12,} ({late_rate:.1%})')
print(f'  Cancelled Orders       : {total_cancelled:>12,} ({cancel_rate:.1%})')
print(f'  Avg Delay              : {avg_delay:>11.2f} days')
print(f'  Revenue At Risk        : ${revenue_at_risk:>12,.0f} ({revenue_at_risk/total_revenue:.1%})')
print()
print(f'  SUPPLIER RISK')
print(f'  Total Regions          : {total_regions:>12}')
print(f'  High Risk Regions      : {high_risk_regions:>12} ({high_risk_regions/total_regions:.1%})')
print()
print(f'  FORECAST')
print(f'  6-Month Demand Growth  : {forecast_growth:>+11.1f}%')
print('=' * 60)

  MASTER KPI SUMMARY — SUPPLY CHAIN INTELLIGENCE
  REVENUE
  Total Revenue          : $  36,784,735
  Total Profit           : $   3,966,903
  Profit Margin          :        10.8%
  Avg Order Value        : $      203.77
  Daily Revenue (avg)    : $      32,669

  DELIVERY PERFORMANCE
  Total Orders           :      180,519
  Late Orders            :       98,977 (54.8%)
  Cancelled Orders       :        3,692 (2.0%)
  Avg Delay              :        0.57 days
  Revenue At Risk        : $  20,126,395 (54.7%)

  SUPPLIER RISK
  Total Regions          :           23
  High Risk Regions      :           12 (52.2%)

  FORECAST
  6-Month Demand Growth  :        +6.2%


---
## 3. Financial Impact of Late Deliveries

In [3]:
# ── Cost model for late deliveries ────────────────────────────
# Industry benchmarks for e-commerce/supply chain:
# - Customer retention loss : 5% of late order revenue lost permanently
# - Expediting costs        : $15 per late order (rush shipping, handling)
# - Customer service cost   : $8 per late order (complaints, refunds)
# - Brand damage (NPS hit)  : estimated 2% revenue impact on late orders

retention_loss_pct   = 0.05
expediting_cost      = 15
customer_service_cost= 8
brand_damage_pct     = 0.02

retention_loss    = revenue_at_risk * retention_loss_pct
expediting_total  = total_late * expediting_cost
cs_total          = total_late * customer_service_cost
brand_damage      = revenue_at_risk * brand_damage_pct
total_cost        = retention_loss + expediting_total + cs_total + brand_damage

print('=' * 60)
print('  FINANCIAL IMPACT OF LATE DELIVERIES')
print('=' * 60)
print(f'  Late Orders            : {total_late:>12,}')
print(f'  Revenue At Risk        : ${revenue_at_risk:>12,.0f}')
print()
print(f'  Cost Breakdown:')
print(f'  Customer Retention Loss: ${retention_loss:>12,.0f}')
print(f'  Expediting Costs       : ${expediting_total:>12,.0f}')
print(f'  Customer Service Costs : ${cs_total:>12,.0f}')
print(f'  Brand Damage Estimate  : ${brand_damage:>12,.0f}')
print(f'  ─────────────────────────────────────────')
print(f'  TOTAL ESTIMATED COST   : ${total_cost:>12,.0f}')
print(f'  Cost as % of Revenue   : {total_cost/total_revenue*100:>11.1f}%')
print('=' * 60)

  FINANCIAL IMPACT OF LATE DELIVERIES
  Late Orders            :       98,977
  Revenue At Risk        : $  20,126,395

  Cost Breakdown:
  Customer Retention Loss: $   1,006,320
  Expediting Costs       : $   1,484,655
  Customer Service Costs : $     791,816
  Brand Damage Estimate  : $     402,528
  ─────────────────────────────────────────
  TOTAL ESTIMATED COST   : $   3,685,319
  Cost as % of Revenue   :        10.0%


---
## 4. ROI Analysis — What Does Fixing This Save?

In [4]:
# ── ROI of implementing the prediction model ───────────────────
# Scenario: Use Week 3 ML model to flag high-risk orders
# and intervene proactively (priority routing, supplier alerts)

model_recall     = 0.652   # from Week 3 Random Forest
intervention_eff = 0.40    # 40% of flagged late orders can be prevented

orders_flagged   = total_late * model_recall
orders_prevented = orders_flagged * intervention_eff
prevention_rate  = orders_prevented / total_orders

# Savings
revenue_saved    = orders_prevented * avg_order_value
cost_saved       = (orders_prevented / total_late) * total_cost
total_savings    = revenue_saved * retention_loss_pct + cost_saved

# Implementation cost estimate
impl_cost        = 50000   # one-time ML system setup
annual_ops_cost  = 12000   # annual maintenance
total_impl_cost  = impl_cost + annual_ops_cost

roi              = (total_savings - total_impl_cost) / total_impl_cost * 100

print('=' * 60)
print('  ROI ANALYSIS — ML INTERVENTION MODEL')
print('=' * 60)
print(f'  Model Recall (Week 3)  : {model_recall:.1%}')
print(f'  Intervention Effect    : {intervention_eff:.0%} of flagged orders saved')
print()
print(f'  Orders Flagged         : {orders_flagged:>12,.0f}')
print(f'  Orders Prevented       : {orders_prevented:>12,.0f}')
print(f'  Late Rate Reduction    : {prevention_rate:.1%}')
print()
print(f'  Revenue Saved          : ${revenue_saved * retention_loss_pct:>12,.0f}')
print(f'  Cost Saved             : ${cost_saved:>12,.0f}')
print(f'  Total Savings          : ${total_savings:>12,.0f}')
print()
print(f'  Implementation Cost    : ${total_impl_cost:>12,.0f}')
print(f'  NET ROI                : {roi:>+11.1f}%')
print('=' * 60)

  ROI ANALYSIS — ML INTERVENTION MODEL
  Model Recall (Week 3)  : 65.2%
  Intervention Effect    : 40% of flagged orders saved

  Orders Flagged         :       64,533
  Orders Prevented       :       25,813
  Late Rate Reduction    : 14.3%

  Revenue Saved          : $     263,001
  Cost Saved             : $     961,131
  Total Savings          : $   1,224,132

  Implementation Cost    : $      62,000
  NET ROI                :     +1874.4%


---
## 5. Business Impact Charts

In [5]:
# ── Chart 1: Revenue breakdown — Safe vs At Risk ───────────────
fig = go.Figure(go.Waterfall(
    name='Revenue Flow',
    orientation='v',
    measure=['absolute', 'relative', 'relative', 'relative', 'total'],
    x=['Total Revenue', 'Revenue at Risk', 'Retention Loss',
       'Operational Costs', 'Net Revenue'],
    y=[total_revenue, -revenue_at_risk * retention_loss_pct,
       -expediting_total - cs_total, -brand_damage,
       0],
    connector={'line': {'color': 'rgb(63, 63, 63)'}},
    increasing={'marker': {'color': '#1D9E75'}},
    decreasing={'marker': {'color': '#E24B4A'}},
    totals={'marker': {'color': '#3B8BD4'}}
))
fig.update_layout(
    title='Revenue Waterfall — Impact of Late Deliveries',
    yaxis_title='Revenue ($)',
    plot_bgcolor='white', height=480
)
fig.write_html('../outputs/impact_revenue_waterfall.html')
fig.show()
print('Chart 1 saved ✓')

Chart 1 saved ✓


In [6]:
# ── Chart 2: Cost breakdown pie ────────────────────────────────
cost_items  = ['Retention Loss', 'Expediting Costs',
               'Customer Service', 'Brand Damage']
cost_values = [retention_loss, expediting_total, cs_total, brand_damage]

fig = px.pie(
    names=cost_items, values=cost_values,
    title=f'Cost Breakdown of Late Deliveries (Total: ${total_cost:,.0f})',
    color_discrete_sequence=['#E24B4A', '#F5A623', '#3B8BD4', '#9B59B6'],
    hole=0.4
)
fig.update_traces(textinfo='label+percent+value',
                  texttemplate='%{label}<br>$%{value:,.0f}<br>(%{percent})')
fig.update_layout(height=480)
fig.write_html('../outputs/impact_cost_breakdown.html')
fig.show()
print('Chart 2 saved ✓')

Chart 2 saved ✓


In [7]:
# ── Chart 3: Late delivery rate by market over time ────────────
market_time = df.groupby(['Order_YearMonth', 'Market'])['Is_Late'].mean().reset_index()
market_time['Late_Rate_Pct'] = market_time['Is_Late'] * 100

fig = px.line(
    market_time,
    x='Order_YearMonth', y='Late_Rate_Pct',
    color='Market',
    title='Late Delivery Rate Trend by Market (Monthly)',
    labels={'Late_Rate_Pct': 'Late Rate (%)', 'Order_YearMonth': 'Month'}
)
fig.update_layout(plot_bgcolor='white', height=450)
fig.write_html('../outputs/impact_late_rate_by_market.html')
fig.show()
print('Chart 3 saved ✓')

Chart 3 saved ✓


In [8]:
# ── Chart 4: Supplier risk vs revenue bubble ───────────────────
color_map = {
    'Low Risk'   : '#1D9E75',
    'Medium Risk': '#F5A623',
    'High Risk'  : '#E24B4A'
}

fig = px.scatter(
    supplier,
    x='Risk_Score', y='Total_Revenue',
    size='Total_Orders',
    color='Risk_Tier',
    color_discrete_map=color_map,
    text='Order Region',
    title='Supplier Risk Score vs Revenue (bubble = order volume)',
    labels={
        'Risk_Score'   : 'Risk Score (0–100)',
        'Total_Revenue': 'Total Revenue ($)',
        'Total_Orders' : 'Order Volume'
    }
)
fig.update_traces(textposition='top center')
fig.update_layout(plot_bgcolor='white', height=520)
fig.write_html('../outputs/impact_supplier_risk_revenue.html')
fig.show()
print('Chart 4 saved ✓')

Chart 4 saved ✓


In [9]:
# ── Chart 5: ROI comparison bar ────────────────────────────────
roi_items  = ['Implementation Cost', 'Total Savings', 'Net Benefit']
roi_values = [total_impl_cost, total_savings, total_savings - total_impl_cost]
roi_colors = ['#E24B4A', '#1D9E75', '#3B8BD4']

fig = go.Figure(go.Bar(
    x=roi_items, y=roi_values,
    marker_color=roi_colors,
    text=[f'${v:,.0f}' for v in roi_values],
    textposition='outside'
))
fig.update_layout(
    title=f'ROI Analysis — ML Intervention Model (ROI: {roi:.0f}%)',
    yaxis_title='Amount ($)',
    plot_bgcolor='white', height=420
)
fig.write_html('../outputs/impact_roi_analysis.html')
fig.show()
print('Chart 5 saved ✓')

Chart 5 saved ✓


In [10]:
# ── Chart 6: Before vs After intervention ─────────────────────
scenarios = ['Current State', 'With ML Intervention']
late_rates = [late_rate * 100, (late_rate - prevention_rate) * 100]
rev_at_risk = [revenue_at_risk, revenue_at_risk - revenue_saved]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Late Delivery Rate (%)', 'Revenue at Risk ($)'))

fig.add_trace(go.Bar(
    x=scenarios, y=late_rates,
    marker_color=['#E24B4A', '#1D9E75'],
    text=[f'{v:.1f}%' for v in late_rates],
    textposition='outside', name='Late Rate'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=scenarios, y=rev_at_risk,
    marker_color=['#E24B4A', '#1D9E75'],
    text=[f'${v:,.0f}' for v in rev_at_risk],
    textposition='outside', name='Revenue at Risk'
), row=1, col=2)

fig.update_layout(
    title='Before vs After ML Intervention',
    plot_bgcolor='white', height=420, showlegend=False
)
fig.write_html('../outputs/impact_before_after.html')
fig.show()
print('Chart 6 saved ✓')

Chart 6 saved ✓


---
## 6. Executive Summary Table

In [11]:
# ── Build executive summary ────────────────────────────────────
exec_summary = pd.DataFrame([
    # Revenue
    {'Category': 'Revenue',    'KPI': 'Total Revenue',          'Value': f'${total_revenue:,.0f}',        'Status': '✅'},
    {'Category': 'Revenue',    'KPI': 'Total Profit',           'Value': f'${total_profit:,.0f}',         'Status': '✅'},
    {'Category': 'Revenue',    'KPI': 'Profit Margin',          'Value': f'{profit_margin:.1f}%',         'Status': '✅'},
    {'Category': 'Revenue',    'KPI': 'Revenue at Risk',        'Value': f'${revenue_at_risk:,.0f}',      'Status': '🔴'},
    # Delivery
    {'Category': 'Delivery',   'KPI': 'Total Orders',           'Value': f'{total_orders:,}',             'Status': '✅'},
    {'Category': 'Delivery',   'KPI': 'Late Delivery Rate',     'Value': f'{late_rate:.1%}',              'Status': '🔴'},
    {'Category': 'Delivery',   'KPI': 'Cancellation Rate',      'Value': f'{cancel_rate:.1%}',            'Status': '🟡'},
    {'Category': 'Delivery',   'KPI': 'Avg Delay',              'Value': f'{avg_delay:.2f} days',         'Status': '🟡'},
    # Supplier
    {'Category': 'Supplier',   'KPI': 'Total Regions',          'Value': f'{total_regions}',              'Status': '✅'},
    {'Category': 'Supplier',   'KPI': 'High Risk Regions',      'Value': f'{high_risk_regions} ({high_risk_regions/total_regions:.0%})', 'Status': '🔴'},
    {'Category': 'Supplier',   'KPI': 'Supplier Rev at Risk',   'Value': f'${revenue_at_risk:,.0f}',      'Status': '🔴'},
    # ML Model
    {'Category': 'ML Model',   'KPI': 'Model ROC-AUC',          'Value': '0.741',                         'Status': '✅'},
    {'Category': 'ML Model',   'KPI': 'Model Recall',           'Value': '65.2%',                         'Status': '✅'},
    {'Category': 'ML Model',   'KPI': 'Orders Preventable',     'Value': f'{orders_prevented:,.0f}',      'Status': '✅'},
    # Forecast
    {'Category': 'Forecast',   'KPI': 'Forecast Accuracy',      'Value': '92.8%',                         'Status': '✅'},
    {'Category': 'Forecast',   'KPI': 'Expected Growth',        'Value': f'{forecast_growth:+.1f}%',      'Status': '✅'},
    # ROI
    {'Category': 'ROI',        'KPI': 'Total Savings',          'Value': f'${total_savings:,.0f}',        'Status': '✅'},
    {'Category': 'ROI',        'KPI': 'Implementation Cost',    'Value': f'${total_impl_cost:,.0f}',      'Status': '✅'},
    {'Category': 'ROI',        'KPI': 'Net ROI',                'Value': f'{roi:.0f}%',                   'Status': '✅'},
])

print('EXECUTIVE SUMMARY')
print('=' * 60)
display(exec_summary)
print('=' * 60)

EXECUTIVE SUMMARY


,Category,KPI,Value,Status
0,Revenue,Total Revenue,"$36,784,735",✅
1,Revenue,Total Profit,"$3,966,903",✅
2,Revenue,Profit Margin,10.8%,✅
3,Revenue,Revenue at Risk,"$20,126,395",🔴
4,Delivery,Total Orders,"180,519",✅
5,Delivery,Late Delivery Rate,54.8%,🔴
6,Delivery,Cancellation Rate,2.0%,🟡
7,Delivery,Avg Delay,0.57 days,🟡
8,Supplier,Total Regions,23,✅
9,Supplier,High Risk Regions,12 (52%),🔴


---
## 7. Save All Outputs

In [12]:
# ── Save executive summary ─────────────────────────────────────
exec_summary.to_csv('../data/processed/executive_summary.csv', index=False)
print('Saved: executive_summary.csv')

# ── Save full business impact report ──────────────────────────
impact_report = pd.DataFrame([{
    'Total_Revenue'         : round(total_revenue, 2),
    'Total_Profit'          : round(total_profit, 2),
    'Profit_Margin_Pct'     : round(profit_margin, 2),
    'Total_Orders'          : total_orders,
    'Late_Orders'           : int(total_late),
    'Late_Rate_Pct'         : round(late_rate * 100, 2),
    'Cancel_Rate_Pct'       : round(cancel_rate * 100, 2),
    'Avg_Delay_Days'        : round(avg_delay, 2),
    'Revenue_At_Risk'       : round(revenue_at_risk, 2),
    'Total_Cost_Late'       : round(total_cost, 2),
    'Cost_Pct_Revenue'      : round(total_cost / total_revenue * 100, 2),
    'High_Risk_Regions'     : int(high_risk_regions),
    'Total_Regions'         : int(total_regions),
    'Model_ROC_AUC'         : 0.741,
    'Model_Recall'          : 0.652,
    'Orders_Preventable'    : round(orders_prevented, 0),
    'Forecast_Accuracy_Pct' : 92.8,
    'Forecast_Growth_Pct'   : round(forecast_growth, 2),
    'Total_Savings'         : round(total_savings, 2),
    'Impl_Cost'             : total_impl_cost,
    'Net_ROI_Pct'           : round(roi, 2)
}])
impact_report.to_csv('../data/processed/business_impact_report.csv', index=False)
print('Saved: business_impact_report.csv')

# ── Save KPI summary for Streamlit app ────────────────────────
kpi_summary = pd.DataFrame([
    {'KPI': 'Total Revenue',       'Value': total_revenue,      'Format': 'currency'},
    {'KPI': 'Revenue at Risk',     'Value': revenue_at_risk,    'Format': 'currency'},
    {'KPI': 'Late Delivery Rate',  'Value': late_rate * 100,    'Format': 'percent'},
    {'KPI': 'High Risk Regions',   'Value': high_risk_regions,  'Format': 'number'},
    {'KPI': 'Total Cost of Delays','Value': total_cost,         'Format': 'currency'},
    {'KPI': 'Orders Preventable',  'Value': orders_prevented,   'Format': 'number'},
    {'KPI': 'Total Savings',       'Value': total_savings,      'Format': 'currency'},
    {'KPI': 'Net ROI',             'Value': roi,                'Format': 'percent'},
    {'KPI': 'Forecast Accuracy',   'Value': 92.8,               'Format': 'percent'},
    {'KPI': 'Forecast Growth',     'Value': forecast_growth,    'Format': 'percent'},
])
kpi_summary.to_csv('../data/processed/kpi_summary.csv', index=False)
print('Saved: kpi_summary.csv')

print()
print('Week 6 complete! ✓')
print('Next step → Week 7: Power BI Dashboard')

Saved: executive_summary.csv
Saved: business_impact_report.csv
Saved: kpi_summary.csv

Week 6 complete! ✓
Next step → Week 7: Power BI Dashboard


---
## Week 6 Summary

| Finding | Value |
|---|---|
| Total Revenue | $36.7M |
| Revenue at Risk | ~$20M (54.8% of revenue) |
| Total Cost of Late Deliveries | Calculated above |
| High Risk Regions | 12 out of 23 (52%) |
| Orders Preventable with ML | ~26% of late orders |
| Forecast Accuracy | 92.8% |
| Net ROI of ML System | Calculated above |

### Key Insight for Portfolio / Interviews
> *"Our end-to-end supply chain analysis identified that late deliveries cost the business an estimated $X in retention loss, expediting costs, and brand damage. By deploying our Random Forest prediction model with 65% recall, we can prevent approximately X,XXX late deliveries, generating a net ROI of XXX% on the implementation investment."*